In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.optimizers import Adam
import seaborn as sns

In [ ]:
datasetPath = './catanddogsmall'

for root, dirs, files in os.walk(datasetPath):
    level = root.replace(datasetPath, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:
        print(f'{subindent}{file}')

In [ ]:
catsDir = os.path.join(datasetPath, 'cats')
dogsDir = os.path.join(datasetPath, 'dogs')

if not os.path.exists(catsDir):
    raise FileNotFoundError(f"Cats directory not found at {catsDir}")
if not os.path.exists(dogsDir):
    raise FileNotFoundError(f"Dogs directory not found at {dogsDir}")

catFiles = [f for f in os.listdir(catsDir) if f.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'))]
dogFiles = [f for f in os.listdir(dogsDir) if f.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'))]

print(f"Number of cat images: {len(catFiles)}")
print(f"Number of dog images: {len(dogFiles)}")
print(f"Total images: {len(catFiles) + len(dogFiles)}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle('Sample Images from Dataset', fontsize=16)

for i in range(4):
    catImg = load_img(os.path.join(catsDir, catFiles[i]), target_size=(150, 150))
    axes[0, i].imshow(catImg)
    axes[0, i].set_title('Cat')
    axes[0, i].axis('off')

for i in range(4):
    dogImg = load_img(os.path.join(dogsDir, dogFiles[i]), target_size=(150, 150))
    axes[1, i].imshow(dogImg)
    axes[1, i].set_title('Dog')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import shutil

trainDir = './data/train'
testDir = './data/test'

os.makedirs(f'{trainDir}/cats', exist_ok=True)
os.makedirs(f'{trainDir}/dogs', exist_ok=True)
os.makedirs(f'{testDir}/cats', exist_ok=True)
os.makedirs(f'{testDir}/dogs', exist_ok=True)

splitRatio = 0.8
splitIdx = int(len(catFiles) * splitRatio)
trainCats = catFiles[:splitIdx]
testCats = catFiles[splitIdx:]

splitIdx = int(len(dogFiles) * splitRatio)
trainDogs = dogFiles[:splitIdx]
testDogs = dogFiles[splitIdx:]

for catFile in trainCats:
    shutil.copy(os.path.join(catsDir, catFile), f'{trainDir}/cats/{catFile}')
for dogFile in trainDogs:
    shutil.copy(os.path.join(dogsDir, dogFile), f'{trainDir}/dogs/{dogFile}')
for catFile in testCats:
    shutil.copy(os.path.join(catsDir, catFile), f'{testDir}/cats/{catFile}')
for dogFile in testDogs:
    shutil.copy(os.path.join(dogsDir, dogFile), f'{testDir}/dogs/{dogFile}')

print(f"Training set - Cats: {len(trainCats)}, Dogs: {len(trainDogs)}")
print(f"Test set - Cats: {len(testCats)}, Dogs: {len(testDogs)}")

In [ ]:
trainDatagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

testDatagen = ImageDataGenerator(rescale=1./255)

trainGenerator = trainDatagen.flow_from_directory(
    trainDir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)

testGenerator = testDatagen.flow_from_directory(
    testDir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

print("Data generators created successfully!")
print(f"Training batches: {len(trainGenerator)}")
print(f"Test batches: {len(testGenerator)}")

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("Model compiled successfully!")

In [ ]:
earlyStopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    trainGenerator,
    steps_per_epoch=len(trainGenerator),
    epochs=20,
    validation_data=testGenerator,
    validation_steps=len(testGenerator),
    callbacks=[earlyStopping]
)

print("Training completed!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
testLoss, testAccuracy = model.evaluate(testGenerator)
print(f"Test Loss: {testLoss:.4f}")
print(f"Test Accuracy: {testAccuracy:.4f}")

In [ ]:
predictions = model.predict(testGenerator)
predictedClasses = (predictions > 0.5).astype('int').flatten()
trueClasses = testGenerator.classes

print("\nClassification Report:")
print(classification_report(trueClasses, predictedClasses, target_names=['Cat', 'Dog']))

cm = confusion_matrix(trueClasses, predictedClasses)
print("\nConfusion Matrix:")
print(cm)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Cat', 'Dog'],
            yticklabels=['Cat', 'Dog'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
def predictImage(imagePath):
    img = load_img(imagePath, target_size=(150, 150))
    imgArray = img_to_array(img) / 255.0
    imgArray = np.expand_dims(imgArray, axis=0)
    
    prediction = model.predict(imgArray)
    confidence = prediction[0][0]
    
    if confidence > 0.5:
        label = 'Dog'
        confidence = confidence
    else:
        label = 'Cat'
        confidence = 1 - confidence
    
    return label, confidence, img

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Predictions on Test Images', fontsize=16)

for i in range(3):
    catPath = os.path.join(testDir, 'cats', testCats[i])
    label, confidence, img = predictImage(catPath)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'{label} ({confidence:.2%})')
    axes[0, i].axis('off')

for i in range(3):
    dogPath = os.path.join(testDir, 'dogs', testDogs[i])
    label, confidence, img = predictImage(dogPath)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'{label} ({confidence:.2%})')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
model.save('cat_dog_classifier.h5')
print("Model saved as 'cat_dog_classifier.h5'")